# Building a multimodal tool-using agent

This tutorial combines image understanding and tool use with LangChain, LangGraph, Groq, and Meta's Llama 4 Scout model.

You will learn how to:

1. Configure `meta-llama/llama-4-scout-17b-16e-instruct` with `ChatGroq`.
2. Send a public image URL to a multimodal model.
3. Read local and remote images as base64 strings and construct data URIs.
4. Create an agent that combines visual reasoning with several typed tools.
5. Inspect the resulting model and tool-call trace.

The example uses a photograph of the **Statue of Liberty and New York Harbor** that appears in Groq's official vision documentation.

## Environment setup

Add `GROQ_API_KEY` to the repository's local `.env` file. The notebook uses only packages already declared in `pyproject.toml`.

In [1]:
import os
import sys

from dotenv import load_dotenv


load_dotenv()

MODEL_NAME = "meta-llama/llama-4-scout-17b-16e-instruct"
assert os.getenv("GROQ_API_KEY"), "Add GROQ_API_KEY to your .env file."

print(f"Python executable: {sys.executable}")
print(f"Selected model: {MODEL_NAME}")

Python executable: /Users/mlstudio/Claude-Code/ml-lab-new/tutorial-agentic-ai/.venv/bin/python3
Selected model: meta-llama/llama-4-scout-17b-16e-instruct


## Create the multimodal model

Llama 4 Scout accepts text and image content and supports tool calling. Groq currently classifies this model as preview, so confirm availability before relying on it in production.

In [2]:
from langchain_groq import ChatGroq


llm = ChatGroq(
    model=MODEL_NAME,
    temperature=0,
    max_tokens=1024,
)

print(f"Model integration: {type(llm).__name__}")

/Users/mlstudio/Claude-Code/ml-lab-new/tutorial-agentic-ai/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Model integration: ChatGroq


## Select an image from the web

The public JPEG is passed directly to the model and is also small enough to demonstrate base64 encoding while remaining below Groq's base64 request-size limit.

Image source: [Groq Images and Vision documentation](https://console.groq.com/docs/vision), hosted by Encyclopaedia Britannica.

In [3]:
from IPython.display import Image, display


IMAGE_URL = (
    "https://cdn.britannica.com/61/93061-050-99147DCE/"
    "Statue-of-Liberty-Island-New-York-Bay.jpg"
)

display(Image(url=IMAGE_URL, width=700))

## Read images as base64

The helpers below support both local files and web URLs. They validate remote content types, enforce a download limit, and return plain base64 strings. `make_image_data_uri` adds the prefix required when a model request embeds the image instead of referencing a public URL.

In [4]:
import base64
import mimetypes
from pathlib import Path
from urllib.request import Request, urlopen


MAX_IMAGE_BYTES = 3 * 1024 * 1024


def read_image_file_as_base64(image_path: str | Path) -> str:
    """Read a local image file and return its base64-encoded contents."""
    path = Path(image_path)
    image_bytes = path.read_bytes()
    if len(image_bytes) > MAX_IMAGE_BYTES:
        raise ValueError(f"Image is larger than {MAX_IMAGE_BYTES:,} bytes.")
    return base64.b64encode(image_bytes).decode("utf-8")


def read_image_url_as_base64(
    image_url: str,
    *,
    timeout: float = 20.0,
) -> tuple[str, str]:
    """Download an image and return `(base64_content, MIME_type)`."""
    request = Request(image_url, headers={"User-Agent": "multimodal-agent-tutorial/1.0"})
    with urlopen(request, timeout=timeout) as response:
        mime_type = response.headers.get_content_type()
        if not mime_type.startswith("image/"):
            raise ValueError(f"Expected an image response, received {mime_type!r}.")
        image_bytes = response.read(MAX_IMAGE_BYTES + 1)

    if len(image_bytes) > MAX_IMAGE_BYTES:
        raise ValueError(f"Image is larger than {MAX_IMAGE_BYTES:,} bytes.")

    encoded = base64.b64encode(image_bytes).decode("utf-8")
    return encoded, mime_type


def make_image_data_uri(base64_image: str, mime_type: str) -> str:
    """Create a data URI suitable for an `image_url` content block."""
    if not mime_type.startswith("image/"):
        raise ValueError("mime_type must be an image MIME type.")
    return f"data:{mime_type};base64,{base64_image}"


def image_file_to_data_uri(image_path: str | Path) -> str:
    """Read a local image and return a complete base64 data URI."""
    path = Path(image_path)
    mime_type, _ = mimetypes.guess_type(path.name)
    if mime_type is None or not mime_type.startswith("image/"):
        raise ValueError(f"Could not infer an image MIME type for {path}.")
    return make_image_data_uri(read_image_file_as_base64(path), mime_type)

In [5]:
base64_image, image_mime_type = read_image_url_as_base64(IMAGE_URL)
image_data_uri = make_image_data_uri(base64_image, image_mime_type)

print(f"MIME type: {image_mime_type}")
print(f"Base64 characters: {len(base64_image):,}")
print(f"Data URI prefix: {image_data_uri[:40]}...")

MIME type: image/jpeg
Base64 characters: 435,508
Data URI prefix: data:image/jpeg;base64,/9j/2wBDAAYEBQYFB...


## Invoke the model with the image URL

A multimodal user message contains a list of content blocks. The text block asks the question, while the `image_url` block supplies the image.

In [6]:
from langchain_core.messages import HumanMessage


vision_message = HumanMessage(
    content=[
        {
            "type": "text",
            "text": (
                "Describe this image in three sentences. Focus on the most "
                "prominent geographic features and avoid guessing hidden details."
            ),
        },
        {
            "type": "image_url",
            "image_url": {"url": IMAGE_URL},
        },
    ]
)

vision_response = llm.invoke([vision_message])
print(vision_response.content)

The image depicts the Statue of Liberty situated on a small island in a body of water, with a large city skyline visible in the background. The Statue of Liberty is positioned on a stone pedestal and base, and the city skyline features numerous tall buildings and skyscrapers. The body of water appears to be a harbor or bay, with the city skyline stretching across the horizon.


## Define tools for the agent

The model can inspect the image itself, but deterministic tools should provide factual metadata and calculations. These tools expose three distinct capabilities: mission metadata lookup, geographic fact lookup, and date arithmetic.

In [7]:
from langchain.tools import tool


LANDMARK_METADATA = {
    "official name": "The landmark's official name is Liberty Enlightening the World.",
    "dedication date": "The Statue of Liberty was dedicated on October 28, 1886.",
    "location": "The statue stands on Liberty Island in New York Harbor, New York City.",
    "designer": "French sculptor Frédéric Auguste Bartholdi designed the statue.",
}

LOCATION_FACTS = {
    "liberty island": "Liberty Island is federally owned and lies within the territorial jurisdiction of New York State.",
    "new york harbor": "New York Harbor connects the Hudson River estuary to the Atlantic Ocean.",
    "new york city": "New York City consists of five boroughs: Manhattan, Brooklyn, Queens, the Bronx, and Staten Island.",
}


@tool
def lookup_landmark_metadata(field: str) -> str:
    """Look up verified Statue of Liberty metadata: official name, dedication date, location, or designer."""
    normalized_field = field.strip().lower()
    value = LANDMARK_METADATA.get(normalized_field)
    if value is None:
        available = ", ".join(sorted(LANDMARK_METADATA))
        return f"Unknown field. Available fields: {available}."
    return value


@tool
def lookup_location_fact(location_name: str) -> str:
    """Return one verified fact about a location associated with the landmark."""
    normalized_name = location_name.strip().lower()
    value = LOCATION_FACTS.get(normalized_name)
    if value is None:
        available = ", ".join(sorted(LOCATION_FACTS))
        return f"Location not found. Available locations: {available}."
    return value


@tool
def calculate_years_since(event_year: int, reference_year: int) -> str:
    """Calculate whole calendar years from an event year through a reference year."""
    if reference_year < event_year:
        return "The reference year must not be earlier than the event year."
    return f"{reference_year - event_year} calendar years"


tools = [
    lookup_landmark_metadata,
    lookup_location_fact,
    calculate_years_since,
]

for current_tool in tools:
    print(f"{current_tool.name}: {current_tool.description}")

lookup_landmark_metadata: Look up verified Statue of Liberty metadata: official name, dedication date, location, or designer.
lookup_location_fact: Return one verified fact about a location associated with the landmark.
calculate_years_since: Calculate whole calendar years from an event year through a reference year.


## Create the multimodal agent

`create_agent` builds the model-and-tools loop. The image remains in the conversation state while the model requests tools and receives their results.

In [8]:
from langchain.agents import create_agent


SYSTEM_PROMPT = """You are a careful multimodal research assistant.
Inspect images directly, but use tools for factual metadata, location facts,
and arithmetic. Do not claim a tool verified something unless you called it.
If the visible evidence is ambiguous, state the uncertainty. Keep the final
answer concise and list the tools that supported it.
"""

agent = create_agent(
    model=llm,
    tools=tools,
    system_prompt=SYSTEM_PROMPT,
)

## Invoke the agent with image and text

This request requires visual inspection plus all three tools. The agent must identify the landmark from the image before it can choose the correct metadata and location-tool arguments.

In [9]:
agent_request = {
    "messages": [
        HumanMessage(
            content=[
                {
                    "type": "text",
                    "text": (
                        "Analyze this photograph and identify the main landmark. "
                        "Then use the available tools to verify its official name, "
                        "dedication date, and location; retrieve one fact about New "
                        "York Harbor; and calculate how many calendar years passed "
                        "from the dedication year through 2026."
                    ),
                },
                {
                    "type": "image_url",
                    "image_url": {"url": IMAGE_URL},
                },
            ]
        )
    ]
}

agent_result = agent.invoke(agent_request)
agent_result["messages"][-1].pretty_print()

================================== Ai Message ==================================

The main landmark in the photograph is the Statue of Liberty. 

The official name of the Statue of Liberty is Liberty Enlightening the World. It was dedicated on October 28, 1886. The statue stands on Liberty Island in New York Harbor, New York City. 

One verified fact about New York Harbor is that it connects the Hudson River estuary to the Atlantic Ocean.

140 calendar years have passed from the dedication year (1886) through 2026. 

The tools that supported this answer are: lookup_landmark_metadata, lookup_location_fact, and calculate_years_since.


## Inspect the agent trace

The returned state contains the original multimodal message, model tool requests, tool results, and final answer.

In [10]:
from langchain_core.messages import AIMessage, ToolMessage


for index, message in enumerate(agent_result["messages"], start=1):
    print(f"{index}. {message.type}")

    if isinstance(message, AIMessage) and message.tool_calls:
        for tool_call in message.tool_calls:
            print(f"   requested: {tool_call['name']}({tool_call['args']})")
    elif isinstance(message, ToolMessage):
        print(f"   returned: {message.content}")
    elif isinstance(message.content, str) and message.content:
        print(f"   content: {message.content}")
    elif isinstance(message.content, list):
        block_types = [block.get("type", "unknown") for block in message.content]
        print(f"   multimodal blocks: {block_types}")

1. human
   multimodal blocks: ['text', 'image_url']
2. ai
   requested: lookup_landmark_metadata({'field': 'official name'})
3. tool
   returned: The landmark's official name is Liberty Enlightening the World.
4. ai
   requested: lookup_landmark_metadata({'field': 'dedication date'})
5. tool
   returned: The Statue of Liberty was dedicated on October 28, 1886.
6. ai
   requested: lookup_landmark_metadata({'field': 'location'})
7. tool
   returned: The statue stands on Liberty Island in New York Harbor, New York City.
8. ai
   requested: lookup_location_fact({'location_name': 'New York Harbor'})
9. tool
   returned: New York Harbor connects the Hudson River estuary to the Atlantic Ocean.
10. ai
   requested: calculate_years_since({'event_year': 1886, 'reference_year': 2026})
11. tool
   returned: 140 calendar years
12. ai
   content: The main landmark in the photograph is the Statue of Liberty. 

The official name of the Statue of Liberty is Liberty Enlightening the World. It was dedic

## Optional: invoke with embedded base64 instead

For a local or non-public image, replace the URL with the generated data URI. Keep the encoded request within the provider's current size limit.

```python
base64_message = HumanMessage(
    content=[
        {"type": "text", "text": "What is visible in this image?"},
        {"type": "image_url", "image_url": {"url": image_data_uri}},
    ]
)
base64_response = llm.invoke([base64_message])
print(base64_response.content)
```

## Key takeaways

- Multimodal messages combine text and image content blocks.
- Public URLs avoid base64 expansion and are preferable when the provider can fetch the image.
- Base64 data URIs support local or private images but require strict size checks.
- Tools should provide deterministic facts or operations; the model should handle visual interpretation.
- Agent traces make it possible to verify which conclusions came from the image and which came from tools.

Provider references:

- [Groq Images and Vision](https://console.groq.com/docs/vision)
- [Groq supported models](https://console.groq.com/docs/models)
- [LangChain ChatGroq integration](https://docs.langchain.com/oss/python/integrations/chat/groq)